# Lab 05 - Blob indexing with Content Understanding

## Required permissions

- **You:** **Search Service Contributor** + **Search Index Data Contributor** on Search and **Storage Blob Data Contributor** on the sample container, as in notebook 04. Sign in with `az login` in the deployment tenant.
- **Search system-assigned managed identity:** **Storage Blob Data Reader** on the container and **Cognitive Services User on the parent Foundry resource** for keyless Content Understanding and embeddings. An embedding-only role is not sufficient for Content Understanding.
- **Facilitator:** provide a supported Foundry region and embedding deployment. Bicep supplies the service-identity grants; attendees need no extra administrator role for this lab.

**Attendee alias:** use your `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). Your index, data source, skillset and indexer use `alice01-cu-*`; uploaded blobs use `alice01/content-understanding/`. Local sample documents and the shared services are unchanged. Aliases prevent naming collisions, not enforce access isolation.

**Goal:** extract a two-page PDF as Markdown, chunk it, generate embeddings and retrieve evidence with page ranges. Allow 45-60 minutes. You can run this lab independently of notebook 04.

```text
Original PDF -> authenticated Blob upload -> Search Blob indexer
  -> Content Understanding (extraction + chunking + page locations)
  -> embedding skill + provenance -> index projections -> separate chunk index
  -> hybrid search + semantic ranking -> evidence with source URL and page range
```

Notebook 04 uses built-in text extraction and Text Split. Here **Content Understanding replaces both extraction and splitting**; there is no second Text Split skill. We use the stable Search REST API `2026-04-01`, not preview semantic chunking or image descriptions. The service returns Markdown content; layout preservation should be inspected, not assumed perfect.

**Before you begin:** deploy the workshop resources in East US, sign in with `az login` in the workshop tenant, and select the workshop Python kernel. You need Search Service Contributor + Search Index Data Contributor on Search and Storage Blob Data Contributor on the sample container. Search's system-assigned identity needs Storage Blob Data Reader on the container and **Cognitive Services User on the parent Foundry resource** for Content Understanding and embedding calls. The Bicep templates already declare those service roles. Blob and Foundry networking must allow the requests.

**Costs and region:** the attached Foundry resource must support Content Understanding; East US is the workshop default. Budget for analysis from the first PDF, embeddings and Search usage; do not assume the general 20-document enrichment allowance covers this skill. A document analysis can time out after five minutes and still incur charges. This small original synthetic PDF contains no personal or confidential data. No extra chat-model deployment, API key, account key or SAS is needed.

## 1. Load configuration

Generated resource settings are refreshed from `.env.azure` on each execution of this cell, overriding stale kernel environment values. Keep manual settings such as `WORKSHOP_PREFIX` in `.env`. All objects use the separate `-cu-` names and Blob folder, so this notebook does not overwrite notebook 04's data.

In [1]:
import json
import os
import re
import time
from datetime import datetime, timezone
from email.utils import format_datetime
from pathlib import Path
from urllib.parse import quote

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure', override=True)
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_STORAGE_RESOURCE_ID',
            'AZURE_STORAGE_BLOB_ENDPOINT', 'AZURE_STORAGE_CONTAINER_NAME',
            'AZURE_AI_ACCOUNT_NAME', 'AZURE_AI_EMBEDDING_DEPLOYMENT_NAME')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Run the deployment first.')
prefix = os.getenv('WORKSHOP_PREFIX') or input('Unique attendee prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
storage_endpoint = os.environ['AZURE_STORAGE_BLOB_ENDPOINT'].rstrip('/')
storage_id = os.environ['AZURE_STORAGE_RESOURCE_ID']
container_name = os.environ['AZURE_STORAGE_CONTAINER_NAME']
foundry_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.services.ai.azure.com"
embedding_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.openai.azure.com"
embedding_deployment = os.environ['AZURE_AI_EMBEDDING_DEPLOYMENT_NAME']
embedding_model = os.getenv('AZURE_AI_EMBEDDING_MODEL_NAME', 'text-embedding-3-small')
embedding_dimensions = 1536
semantic_configuration = 'chunk-semantic'
api_version = '2026-04-01'
index_name = f'{prefix}-cu-chunks'
data_source_name = f'{prefix}-cu-source'
skillset_name = f'{prefix}-cu-skills'
indexer_name = f'{prefix}-cu-indexer'
blob_prefix = f'{prefix}/content-understanding/'
credential = AzureCliCredential(tenant_id=os.environ['AZURE_TENANT_ID'])
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
storage_token = get_bearer_token_provider(credential, 'https://storage.azure.com/.default')
print('Search:', search_endpoint)
print('Foundry enrichment:', foundry_endpoint)
print('Index:', index_name, '| source folder:', f'{container_name}/{blob_prefix}')

Search: https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net
Foundry enrichment: https://iq-workshop-ai-eastus-suy43pebejwnk.services.ai.azure.com
Index: megan02-cu-chunks | source folder: workshop-documents/megan02/content-understanding/


## 2. Upload the original PDF

Open [`travel-handbook.pdf`](../data/search/travel-handbook.pdf) first. Page 1 includes a receipt-requirements table; page 2 includes exceptions and an example. This is a fictional policy, not advice or a real employer policy. The upload uses `application/pdf`, Entra authentication and your own folder. Repeating it overwrites only this sample blob.

In [2]:
sample_files = [root / 'data' / 'search' / 'travel-handbook.pdf']
for path in sample_files:
    data = path.read_bytes()
    if not data.startswith(b'%PDF-'):
        raise ValueError(f'Expected a PDF file: {path.name}')
    blob_name = f'{blob_prefix}{path.name}'
    response = requests.put(
        f'{storage_endpoint}/{container_name}/{quote(blob_name, safe="/")}',
        headers={
            'Authorization': f'Bearer {storage_token()}',
            'x-ms-version': '2023-11-03',
            'x-ms-date': format_datetime(datetime.now(timezone.utc), usegmt=True),
            'x-ms-blob-type': 'BlockBlob', 'Content-Type': 'application/pdf',
        }, data=data, timeout=(10, 90),
    )
    if response.status_code >= 400:
        print(response.text)
    response.raise_for_status()
    print('Uploaded:', blob_name)

Uploaded: megan02/content-understanding/travel-handbook.pdf


## 3. Create the Blob data source

The `ResourceId` connection uses Search's system-assigned managed identity, not your notebook token. The virtual folder limits which files are selected, but is not a participant security boundary. Do not skip this cell: an indexer cannot reference a data source that does not exist. Source deletion detection is deliberately not configured; deleting a blob alone does not remove previously indexed chunks.

In [3]:
data_source = {
    'name': data_source_name, 'type': 'azureblob',
    'credentials': {'connectionString': f'ResourceId={storage_id};'},
    'container': {'name': container_name, 'query': blob_prefix},
}
response = requests.put(
    f'{search_endpoint}/datasources/{data_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=data_source, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
print('Data source ready:', data_source_name)

Data source ready: megan02-cu-source


## 4. Define the chunk index

Search projections generate `chunk_id` and `parent_id`; do not map these manually. Each chunk stores Markdown, a 1,536-dimensional embedding, filename/Blob URL and Content Understanding location metadata. `page_from` and `page_to` form a **range**, because a chunk may span pages. `ordinal_position` is ordering metadata, not a page number.

In [4]:
index = {
    'name': index_name,
    'fields': [
        {'name': 'chunk_id', 'type': 'Edm.String', 'key': True,
         'searchable': True, 'analyzer': 'keyword'},
        {'name': 'parent_id', 'type': 'Edm.String', 'filterable': True},
        {'name': 'chunk', 'type': 'Edm.String', 'searchable': True},
        {'name': 'title', 'type': 'Edm.String', 'searchable': True},
        {'name': 'page_from', 'type': 'Edm.Int32', 'filterable': True},
        {'name': 'page_to', 'type': 'Edm.Int32', 'filterable': True},
        {'name': 'ordinal_position', 'type': 'Edm.Int32', 'sortable': True},
        {'name': 'location_source', 'type': 'Edm.String'},
        {'name': 'text_vector', 'type': 'Collection(Edm.Single)',
         'searchable': True, 'retrievable': True, 'dimensions': embedding_dimensions,
         'vectorSearchProfile': 'chunk-vector-profile'},
        {'name': 'source', 'type': 'Edm.ComplexType', 'fields': [
            {'name': 'file_name', 'type': 'Edm.String', 'searchable': True},
            {'name': 'url', 'type': 'Edm.String'},
        ]},
    ],
    'vectorSearch': {
        'algorithms': [{'name': 'chunk-hnsw', 'kind': 'hnsw',
                        'hnswParameters': {'metric': 'cosine'}}],
        'profiles': [{'name': 'chunk-vector-profile', 'algorithm': 'chunk-hnsw',
                      'vectorizer': 'chunk-vectorizer'}],
        'vectorizers': [{
            'name': 'chunk-vectorizer', 'kind': 'azureOpenAI',
            'azureOpenAIParameters': {
                'resourceUri': embedding_endpoint, 'deploymentId': embedding_deployment,
                'modelName': embedding_model,
            },
        }],
    },
    'semantic': {
        'defaultConfiguration': semantic_configuration,
        'configurations': [{
            'name': semantic_configuration,
            'prioritizedFields': {
                'titleField': {'fieldName': 'title'},
                'prioritizedContentFields': [{'fieldName': 'chunk'}],
            },
        }],
    },
}
response = requests.put(
    f'{search_endpoint}/indexes/{index_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, json=index, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
print('Index ready:', index_name)

Index ready: megan02-cu-chunks


## 5. Extract, chunk and embed with managed identities

`file_data` provides the original PDF. Content Understanding produces `/document/text_sections/*` objects whose `content` is Markdown. The embedding skill reads each object's **content**, not the whole object. Shaper preserves the original filename and URL.

`cognitiveServices` attaches the Foundry resource for keyless analysis and billing. `identity=None` selects Search's system identity. The same identity calls embeddings when `apiKey` and `authIdentity` are omitted. Fixed-size chunking uses 1,000 characters and 200-character overlap; preview semantic chunking and image descriptions are intentionally not enabled.

In [5]:
skillset = {
    'name': skillset_name,
    'description': 'Content Understanding PDF extraction, chunking, vectors and page provenance.',
    'cognitiveServices': {
        '@odata.type': '#Microsoft.Azure.Search.AIServicesByIdentity',
        'subdomainUrl': foundry_endpoint, 'identity': None,
    },
    'skills': [
        {
            '@odata.type': '#Microsoft.Skills.Util.ContentUnderstandingSkill',
            'name': 'understand-pdf', 'context': '/document',
            'extractionOptions': ['locationMetadata'],
            'chunkingProperties': {'unit': 'characters', 'maximumLength': 1000, 'overlapLength': 200},
            'inputs': [{'name': 'file_data', 'source': '/document/file_data'}],
            'outputs': [{'name': 'text_sections', 'targetName': 'text_sections'}],
        },
        {
            '@odata.type': '#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill',
            'name': 'embed-sections', 'context': '/document/text_sections/*',
            'resourceUri': embedding_endpoint, 'deploymentId': embedding_deployment,
            'modelName': embedding_model, 'dimensions': embedding_dimensions,
            'inputs': [{'name': 'text', 'source': '/document/text_sections/*/content'}],
            'outputs': [{'name': 'embedding', 'targetName': 'text_vector'}],
        },
        {
            '@odata.type': '#Microsoft.Skills.Util.ShaperSkill',
            'name': 'shape-provenance', 'context': '/document',
            'inputs': [
                {'name': 'file_name', 'source': '/document/metadata_storage_name'},
                {'name': 'url', 'source': '/document/metadata_storage_path'},
            ],
            'outputs': [{'name': 'output', 'targetName': 'provenance'}],
        },
    ],
    'indexProjections': {
        'selectors': [{
            'targetIndexName': index_name, 'parentKeyFieldName': 'parent_id',
            'sourceContext': '/document/text_sections/*',
            'mappings': [
                {'name': 'chunk', 'source': '/document/text_sections/*/content'},
                {'name': 'text_vector', 'source': '/document/text_sections/*/text_vector'},
                {'name': 'title', 'source': '/document/metadata_storage_name'},
                {'name': 'source', 'source': '/document/provenance'},
                {'name': 'page_from', 'source': '/document/text_sections/*/locationMetadata/pageNumberFrom'},
                {'name': 'page_to', 'source': '/document/text_sections/*/locationMetadata/pageNumberTo'},
                {'name': 'ordinal_position', 'source': '/document/text_sections/*/locationMetadata/ordinalPosition'},
                {'name': 'location_source', 'source': '/document/text_sections/*/locationMetadata/source'},
            ],
        }],
        'parameters': {'projectionMode': 'skipIndexingParentDocuments'},
    },
}
response = requests.put(
    f'{search_endpoint}/skillsets/{skillset_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, json=skillset, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
print('Skillset ready:', skillset_name)

Skillset ready: megan02-cu-skills


## 6. Run the Blob indexer

**`allowSkillsetToReadFileData=True` is essential:** it exposes `/document/file_data` to Content Understanding. `dataToExtract='allMetadata'` leaves content extraction to the skill while retaining Blob metadata. The indexer selects PDFs only. Creation starts a run; updating an existing idle indexer explicitly starts a new one. Zero failed-item tolerance prevents partial indexing from being reported as success.

In [6]:
response = requests.get(
    f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
indexer_exists = response.status_code != 404
previous_run_start = None
if indexer_exists:
    response.raise_for_status()
    previous_run = response.json().get('lastResult') or {}
    if previous_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait before updating it.')
    previous_run_start = previous_run.get('startTime')
indexer = {
    'name': indexer_name, 'dataSourceName': data_source_name,
    'targetIndexName': index_name, 'skillsetName': skillset_name,
    'parameters': {
        'maxFailedItems': 0, 'maxFailedItemsPerBatch': 0,
        'configuration': {
            'dataToExtract': 'allMetadata', 'allowSkillsetToReadFileData': True,
            'indexedFileNameExtensions': '.pdf',
        },
    },
}
response = requests.put(
    f'{search_endpoint}/indexers/{indexer_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, json=indexer, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
print('Indexer ready:', indexer_name)
if indexer_exists:
    response = requests.post(
        f'{search_endpoint}/indexers/{indexer_name}/run?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    print('New run requested for the existing indexer.')

Indexer ready: megan02-cu-indexer


In [7]:
for attempt in range(90):
    response = requests.get(
        f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    status = response.json()
    last_run = status.get('lastResult') or {}
    state = last_run.get('status')
    print('Run:', state, '| processed:', last_run.get('itemsProcessed'),
          '| failed:', last_run.get('itemsFailed'))
    is_requested_run = previous_run_start is None or last_run.get('startTime') != previous_run_start
    if state == 'success' and is_requested_run:
        print('Warnings:', json.dumps(last_run.get('warnings', []), indent=2))
        if last_run.get('itemsFailed', 0) or last_run.get('errors'):
            raise RuntimeError(json.dumps(last_run, indent=2))
        break
    if (is_requested_run and state in ('transientFailure', 'persistentFailure', 'error')) or status.get('status') == 'error':
        raise RuntimeError(json.dumps(last_run or status, indent=2))
    if attempt < 89:
        time.sleep(10)
else:
    raise TimeoutError('Indexer did not complete within 15 minutes. Inspect its status before retrying.')

Run: inProgress | processed: 0 | failed: 0
Run: inProgress | processed: 0 | failed: 0
Run: success | processed: 1 | failed: 0
Warnings: []


## 7. Inspect Markdown, page ranges and vectors

Expect several chunks from the one PDF. Compare the receipt table with page 1 and the missing-receipt rules with page 2. A range such as 1-2 means a chunk spans both pages; do not label it as a single page. Inspect the PDF to verify that extracted text and table structure are accurate. The Blob URL still requires authenticated access.

In [8]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={'search': '*', 'count': True, 'top': 100, 'orderby': 'ordinal_position',
          'select': 'chunk_id,parent_id,title,chunk,source,page_from,page_to,ordinal_position'},
    timeout=(10, 90),
)
response.raise_for_status()
chunks = response.json()['value']
if not chunks:
    raise RuntimeError('No chunks were indexed. Inspect the indexer errors and source folder.')
if any(not isinstance(item.get('page_from'), int) or not isinstance(item.get('page_to'), int)
       or not 1 <= item['page_from'] <= item['page_to'] <= 2 for item in chunks):
    raise RuntimeError('Expected valid page ranges from the two-page sample; inspect locationMetadata.')
print('Total chunks:', response.json()['@odata.count'])
print(json.dumps(chunks, indent=2))

Total chunks: 5
[
  {
    "@search.score": 1.0,
    "chunk_id": "0a541620054d_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL21lZ2FuMDIvY29udGVudC11bmRlcnN0YW5kaW5nL3RyYXZlbC1oYW5kYm9vay5wZGY1_text_sections_0",
    "parent_id": "aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL21lZ2FuMDIvY29udGVudC11bmRlcnN0YW5kaW5nL3RyYXZlbC1oYW5kYm9vay5wZGY1",
    "chunk": "Workshop Travel Handbook - ReceiptsOriginal synthetic sample for the Azure IQ Workshop. This is not a real employer policy. Version 1.0. The sample contains two pages so learners can inspect extracted page ranges, Markdown tables, overlapping chunks and source attribution.1. Standard claims. Submit a complete travel claim within 30 calendar days after the trip ends. A claim includes the business purpose, trip dates, cost center and the currency actually charged. Record one expense per line; do not combine meals, transport and lodging.An itemized rece

In [9]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={'search': '*', 'select': 'chunk_id,text_vector', 'top': 3}, timeout=(10, 90),
)
response.raise_for_status()
vector_samples = response.json()['value']
if not vector_samples or any(len(item.get('text_vector') or []) != embedding_dimensions
                             for item in vector_samples):
    raise RuntimeError('Missing or wrong-sized vectors. Inspect embedding skill errors.')
for item in vector_samples:
    print(item['chunk_id'], '| vector dimensions:', len(item['text_vector']))

0a541620054d_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL21lZ2FuMDIvY29udGVudC11bmRlcnN0YW5kaW5nL3RyYXZlbC1oYW5kYm9vay5wZGY1_text_sections_2 | vector dimensions: 1536
0a541620054d_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL21lZ2FuMDIvY29udGVudC11bmRlcnN0YW5kaW5nL3RyYXZlbC1oYW5kYm9vay5wZGY1_text_sections_1 | vector dimensions: 1536
0a541620054d_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL21lZ2FuMDIvY29udGVudC11bmRlcnN0YW5kaW5nL3RyYXZlbC1oYW5kYm9vay5wZGY1_text_sections_3 | vector dimensions: 1536


## 8. Retrieve evidence with hybrid search and semantic ranking

Search generates the query embedding using the index's Azure OpenAI vectorizer. Hybrid retrieval combines keyword and vector candidates; semantic ranking reranks them and produces extractive captions, not an LLM-written answer. Explain the receipt requirements using retrieved text and cite the filename and page range. Check every claim against the PDF.

In [10]:
question = 'Which travel expenses always need itemized receipts, and what if a hotel receipt is missing?'
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'search': question, 'searchFields': 'chunk',
        'vectorQueries': [{'kind': 'text', 'text': question, 'fields': 'text_vector', 'k': 50}],
        'queryType': 'semantic', 'semanticConfiguration': semantic_configuration,
        'captions': 'extractive', 'select': 'chunk_id,title,chunk,source,page_from,page_to', 'top': 3,
    }, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
if response.status_code != 200 or response.json().get('@search.semanticPartialResponseReason'):
    raise RuntimeError(f'Semantic ranking did not complete: {response.json()}')
if not response.json().get('value'):
    raise RuntimeError('No evidence returned. Inspect the indexed chunks before continuing.')
print(json.dumps(response.json(), indent=2))

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes('megan02-cu-chunks')/$metadata#docs(*)",
  "@search.answers": [],
  "value": [
    {
      "@search.score": 0.032522473484277725,
      "@search.rerankerScore": 2.972663402557373,
      "@search.captions": [
        {
          "text": "  Expense category Itemized receipt Extra requirement   Hotel Always required Separate room, tax and meals   Airline or rail Always required Include ticket and itinerary   Meals Required above USD 25 List attendees and purpose   Local taxi Required above USD 15 Record start and destination  Workshop Travel Handbook - ExceptionsOriginal synthetic.",
          "highlights": "  Expense category Itemized receipt Extra requirement   Hotel Always required<em> Separate room, tax and meals   Airline or rail </em>Always required Include<em> ticket and itinerary   Meals Required above USD </em>25 List attendees and purpose   Local taxi Required above USD 15 Record st

## 9. Experiments and reprocessing

Compare these chunks with notebook 04's TXT chunks: which retain table structure and physical page locations? Then change `chunkingProperties.maximumLength` to 2,000 and `overlapLength` to 300. Rerun the skillset cell and explicitly reset/run below, then rerun monitoring and inspection. Updating a skillset alone does not reprocess unchanged files. Resetting reanalyzes the PDF and regenerates embeddings, incurring more charges.

For your own documents, adjust the two-page validation in section 7. Use only non-sensitive PDFs and do not assume image descriptions are enabled by this stable-API lab.

In [ ]:
RUN_AGAIN = False
RESET_BEFORE_RUN = False
if RUN_AGAIN:
    response = requests.get(
        f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    last_run = response.json().get('lastResult') or {}
    if last_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait for it to finish.')
    previous_run_start = last_run.get('startTime')
    if RESET_BEFORE_RUN:
        response = requests.post(
            f'{search_endpoint}/indexers/{indexer_name}/reset?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        response.raise_for_status()
    response = requests.post(
        f'{search_endpoint}/indexers/{indexer_name}/run?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    print('Run requested. Rerun the monitor and result cells.')
else:
    print('No extra run requested. Set RUN_AGAIN=True for the exercise.')

## 10. Optional cleanup

Cleanup is off by default. It deletes only this notebook's four prefixed Search objects and the one uploaded PDF. It does not delete notebook 04's objects, other blobs, the container or shared Azure services. Blob soft deletion may retain billable copies.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for collection, name in [('indexers', indexer_name), ('skillsets', skillset_name),
                             ('indexes', index_name), ('datasources', data_source_name)]:
        response = requests.delete(
            f'{search_endpoint}/{collection}/{name}?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Deleted or already absent:', name)
    for path in sample_files:
        blob_name = f'{blob_prefix}{path.name}'
        response = requests.delete(
            f'{storage_endpoint}/{container_name}/{quote(blob_name, safe="/")}',
            headers={'Authorization': f'Bearer {storage_token()}', 'x-ms-version': '2023-11-03'},
            timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Deleted or already absent:', blob_name)
else:
    print('Cleanup disabled.')

## References and next lab

- [Content Understanding skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-content-understanding)
- [Keyless skillset attachment and billing](https://learn.microsoft.com/azure/search/cognitive-search-attach-cognitive-services)
- [Content Understanding regions](https://learn.microsoft.com/azure/ai-services/content-understanding/language-region-support)
- [Index projections](https://learn.microsoft.com/azure/search/search-how-to-define-index-projections)
- [Azure OpenAI embedding skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-azure-openai-embedding)

Next: [notebook 06](06-push-chunks-vectors.ipynb) chunks TXT files and generates embeddings in Python, then pushes them to a separate index without a Blob indexer.